With sampling tokens / text during work

Option B: language detection inside the worker processes, so both language detection and tokenization run in parallel.

Key points:

The main process only streams and groups raw rows into chunks (no language filtering).

Each worker instantiates its own langid + tiktoken encoder, filters its chunk for English, tokenizes kept rows, saves a .npy chunk file, and returns a small verification sample (first up to 20 token ids + decoded text).

The main process logs each chunk's sample as soon as a worker returns it and then merges all .npy chunk files into one memmapped .npy.

In [1]:
#!/usr/bin/env python3
"""
Parallel parquet -> filter (in-worker) -> tokenize -> chunk-save pipeline (Option B).

- Streams parquet rows with pyarrow.dataset.
- Groups raw rows into text chunks up to TARGET_CHUNK_BYTES (approx bytes).
- Dispatches chunks in parallel to worker processes.
- Each worker:
    * instantiates its own langid + tiktoken encoder
    * filters chunk rows for English
    * tokenizes the kept rows
    * saves tokens as .npy and returns a small sample (first up to 20 tokens and decoded text)
- Main process logs samples and merges all chunk .npy files into one memmapped .npy.
"""

import os
import time
import uuid
from pathlib import Path
import re
import gc
import logging
from typing import List, Tuple, Optional, Iterator

import numpy as np
from joblib import Parallel, delayed, parallel_backend

import pyarrow.dataset as ds
import pyarrow as pa

# NOTE: we import langid and tiktoken at top-level so they are available for workers,
# but the worker will instantiate tokenizer inside the function to avoid pickling problems.
import langid
import tiktoken

# ----------------- Configuration (tweak me) -----------------
INPUT_DIR = "datasets/4GB"                  # parquet directory (absolute or relative)
PARQUET_GLOB = "*.parquet"                 # per-file glob pattern
OUTPUT_DIR = "outputs/extract_v11"         # where chunk .npy files will be saved
MERGED_OUTPUT = os.path.join(OUTPUT_DIR, "tokens_merged.npy")

# Streaming / chunking
BATCH_ROWS = 4096                           # pyarrow rows per RecordBatch
TARGET_CHUNK_BYTES = 50_000_000             # approx UTF-8 bytes per chunk (tune to memory)

# Parallelism
TOKENIZER_MODEL = "gpt2"
N_JOBS = min(max(os.cpu_count(), 1), 128)   # worker processes
PREFETCH_FACTOR = 4                         # how many chunks to prefetch per dispatch
PARALLEL_VERBOSE = 10                       # joblib verbosity (0,5,10 as needed)

# Language filter thresholds (used inside worker)
ASCII_THRESHOLD = 0.5
LANG_MODE = "accuracy"                      # "accuracy" uses langid + ascii fallback

# Save settings
os.makedirs(OUTPUT_DIR, exist_ok=True)


# -- Logging Configuration ----------------------------------------------
run_id = int(time.time())

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S", 
    encoding='utf-8'
)
logger = logging.getLogger(__name__)

# Ensure logs directory exists
LOG_DIR = os.path.join("logs")
if not os.path.exists(LOG_DIR):
    os.makedirs(LOG_DIR)

# Create file handler
log_file = os.path.join(LOG_DIR, f"extract_{run_id}.log")
file_handler = logging.FileHandler(log_file, mode='w', encoding='utf-8')
file_handler.setLevel(logging.DEBUG)

# Optional: use same format as console
formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", "%H:%M:%S")
file_handler.setFormatter(formatter)

# Add handler to logger
logger.addHandler(file_handler)
logger.info(f"Logging to file: {log_file}")
# ----------------- Helpers -----------------

def salvage_stale_temp_files(out_dir: str) -> int:
    """
    Repair leftovers from earlier buggy runs where np.save appended '.npy' to our temp paths.
    Renames:
      '*.npy.tmp.npy' -> '*.npy'
      '*.npy.tmp'     -> '*.npy'
    Returns number of files repaired.
    """
    d = Path(out_dir)
    repaired = 0
    # case 1: .../chunk_xxx.npy.tmp.npy  -> .../chunk_xxx.npy
    for p in d.glob("chunk_*.npy.tmp.npy"):
        final = Path(str(p).replace(".npy.tmp.npy", ".npy"))
        if not final.exists():
            try:
                os.replace(str(p), str(final))
                repaired += 1
            except Exception as e:
                logger.debug(f"Could not salvage {p}: {e}")
    # case 2: .../chunk_xxx.npy.tmp      -> .../chunk_xxx.npy
    for p in d.glob("chunk_*.npy.tmp"):
        final = p.with_suffix("")  # drop '.tmp' -> back to '.npy'
        if not final.exists():
            try:
                os.replace(str(p), str(final))
                repaired += 1
            except Exception as e:
                logger.debug(f"Could not salvage {p}: {e}")
    if repaired:
        logger.info(f"Salvaged {repaired} stale temp files in {out_dir}")
    return repaired


def list_parquet_files(input_dir: str, pattern: str = "*.parquet") -> List[str]:
    p = Path(input_dir)
    if not p.exists():
        raise FileNotFoundError(f"Input dir not found: {p}")
    return sorted(str(x) for x in p.glob(pattern))


def choose_string_column(dataset: ds.dataset) -> str:
    schema = dataset.schema
    str_columns = [f.name for f in schema if pa.types.is_string(f.type)]
    for candidate in ("text", "content", "body"):
        if candidate in str_columns:
            return candidate
    if not str_columns:
        raise RuntimeError("No string column in parquet files")
    return str_columns[0]


# The language detection helper is safe to call both in main and worker contexts.
def is_english_text(text: str, mode: str = "accuracy", ascii_threshold: float = 0.5) -> bool:
    if not text:
        return False
    if mode == "accuracy":
        try:
            lang, score = langid.classify(text)
        except Exception:
            lang = None
        if lang == "en":
            return True
        ascii_count = sum(1 for c in text if ord(c) < 128)
        total = len(text)
        if total == 0:
            return False
        return (ascii_count / total) >= ascii_threshold
    else:
        ascii_count = sum(1 for c in text if ord(c) < 128)
        return (ascii_count / max(1, len(text))) >= ascii_threshold


# ----------------- Streaming & chunking (main process) -----------------
def iter_raw_text_chunks_from_parquets(
    input_dir: str,
    pattern: str = "*.parquet",
    batch_rows: int = 4096,
    target_chunk_bytes: int = 50_000_000,
) -> Iterator[Tuple[List[str], int]]:
    """
    Stream parquet rows (no language filtering) and accumulate raw text rows into
    chunks (list[str]) up to target_chunk_bytes (approx). Yield (list_of_rows, chunk_id).
    """
    file_list = list_parquet_files(input_dir, pattern)
    if not file_list:
        logger.warning("No parquet files found.")
        return

    dataset = ds.dataset(file_list, format="parquet")
    chosen_col = choose_string_column(dataset)
    logger.info(f"Using string column: {chosen_col}")

    scanner = dataset.scanner(batch_size=batch_rows)
    chunk_id = 0
    buffer_texts: List[str] = []
    buffer_bytes = 0

    for record_batch in scanner.to_batches():
        arr = record_batch[chosen_col]  # column by name
        pylist = arr.to_pylist()
        for raw in pylist:
            if raw is None:
                continue
            text = str(raw).strip()
            if not text:
                continue
            b = len(text.encode("utf-8"))
            # if adding this would exceed target and buffer has content -> yield current buffer
            if buffer_texts and (buffer_bytes + b) >= target_chunk_bytes:
                yield buffer_texts, chunk_id
                chunk_id += 1
                buffer_texts = []
                buffer_bytes = 0
            # if single item too big, send it alone
            if b >= target_chunk_bytes:
                yield [text], chunk_id
                chunk_id += 1
                continue
            buffer_texts.append(text)
            buffer_bytes += b

    # flush leftover
    if buffer_texts:
        yield buffer_texts, chunk_id


# ----------------- Worker: filter (in-worker) + tokenize + save -----------------
def _safe_atomic_save_npy(arr: np.ndarray, out_path: Path, max_retries: int = 5, retry_delay: float = 0.1):
    """
    Save `arr` to a unique temporary file in the same directory as out_path,
    then atomically replace out_path with that temp file. Retries on failure.
    NOTE: We open a file handle so NumPy does NOT append '.npy' to our temp path.
    """
    out_dir = out_path.parent
    out_dir.mkdir(parents=True, exist_ok=True)

    # unique temp name in the same directory to ensure atomic replace
    tmp_name = f"{out_path.stem}.{os.getpid()}.{uuid.uuid4().hex}.npy.tmp"
    tmp_path = out_dir / tmp_name

    # Write numpy file to tmp_path via an open file handle to avoid extension munging
    with open(tmp_path, "wb") as f:
        np.save(f, arr)

    # Ensure it exists
    if not tmp_path.exists():
        raise FileNotFoundError(f"Temp save failed, file not found: {tmp_path}")

    # Now attempt atomic replace; retry on transient Windows AV/locking
    last_exc = None
    for attempt in range(1, max_retries + 1):
        try:
            os.replace(str(tmp_path), str(out_path))  # atomic on Windows & POSIX
            return True
        except (FileNotFoundError, PermissionError) as e:
            last_exc = e
            time.sleep(retry_delay)
            continue
        except Exception as e:
            raise
    # If we get here, all retries failed
    raise last_exc or RuntimeError("Atomic replace failed for unknown reason")



def tokenize_filter_and_save(
    chunk_texts: List[str],
    chunk_id: int,
    out_dir: str,
    tokenizer_model: str,
    sample_n: int = 10,
    lang_mode: str = "accuracy",
    ascii_threshold: float = 0.5,
) -> Optional[Tuple[str, int, List[int], str]]:
    """
    Worker: filter rows for English then tokenize and save using safe atomic write.
    """

    # prepare encoder in worker
    try:
        try:
            enc = tiktoken.get_encoding(tokenizer_model)
        except Exception:
            enc = tiktoken.encoding_for_model(tokenizer_model)
    except Exception as e:
        logger.exception(f"Worker failed to create encoder: {e}")
        return None

    # filter rows locally
    kept_rows = []
    for txt in chunk_texts:
        if not txt:
            continue
        if is_english_text(txt, mode=lang_mode, ascii_threshold=ascii_threshold):
            kept_rows.append(txt)
    if not kept_rows:
        return None

    joined = "\n".join(kept_rows)

    # tokenize
    try:
        tokens = enc.encode_ordinary(joined)
    except Exception:
        tokens = enc.encode(joined)

    arr = np.array(tokens, dtype=np.int32)

    out_path = Path(out_dir) / f"chunk_{chunk_id:06d}.npy"

    # If final file already exists, skip writing (resumability)
    if out_path.exists():
        logger.info(f"Chunk file already exists, skipping write: {out_path}")
    else:
        # Robust atomic save
        try:
            _safe_atomic_save_npy(arr, out_path, max_retries=8, retry_delay=0.2)
        except Exception as e:
            logger.exception(f"Failed to atomically save chunk {chunk_id} to {out_path}: {e}")
            # Best-effort cleanup of any temp variants
            try:
                stem = out_path.stem  # e.g., 'chunk_000123'
                patterns = [
                    f"{stem}.*.npy.tmp",
                    f"{stem}.*.npy.tmp.npy",
                    f"{stem}.*.tmp",
                ]
                for pat in patterns:
                    for fpath in out_path.parent.glob(pat):
                        try:
                            fpath.unlink(missing_ok=True)
                        except Exception:
                            pass
            except Exception:
                pass
            return None

    # prepare sample
    sample_tokens = arr[:sample_n].tolist()
    try:
        sample_text = enc.decode(sample_tokens)
    except Exception:
        sample_text = " ".join(str(t) for t in sample_tokens)

    return str(out_path), int(arr.size), sample_tokens, sample_text


# ----------------- Merge chunk files (main process) -----------------
def merge_chunks(chunk_files: List[str], out_path: str):
    """
    Merge numeric-sorted chunk .npy files into a single memmapped .npy (int32).
    """
    if not chunk_files:
        raise ValueError("No chunk files to merge")

    total = 0
    lengths = []
    for f in chunk_files:
        arr = np.load(f, mmap_mode="r")
        lengths.append(arr.shape[0])
        total += arr.shape[0]

    dtype = np.int32
    merged = np.memmap(out_path, dtype=dtype, mode="w+", shape=(total,))
    cursor = 0
    for f in chunk_files:
        arr = np.load(f)
        n = arr.shape[0]
        merged[cursor: cursor + n] = arr
        cursor += n
    merged.flush()
    del merged
    logger.info(f"Merged {len(chunk_files)} chunks ({total} tokens) -> {out_path}")


# ----------------- Orchestration (main) -----------------
def run_pipeline():
    logger.info(f"Starting pipeline (Option B): INPUT={INPUT_DIR}, OUTPUT={OUTPUT_DIR}, N_JOBS={N_JOBS}")

    # Salvage any prior temp artifacts so we don't lose work
    salvage_stale_temp_files(OUTPUT_DIR)

    chunk_generator = iter_raw_text_chunks_from_parquets(
        INPUT_DIR, PARQUET_GLOB, batch_rows=BATCH_ROWS, target_chunk_bytes=TARGET_CHUNK_BYTES
    )

    saved_files = []
    total_tokens = 0

    MAX_PENDING = max(1, N_JOBS * PREFETCH_FACTOR)

    try:
        while True:
            # prefetch up to MAX_PENDING chunks
            to_dispatch = []
            for _ in range(MAX_PENDING):
                try:
                    chunk_texts, chunk_id = next(chunk_generator)
                    to_dispatch.append((chunk_texts, chunk_id))
                except StopIteration:
                    break

            if not to_dispatch:
                # nothing new to dispatch
                break

            logger.info(f"Dispatching {len(to_dispatch)} chunks to workers (n_jobs={N_JOBS})")
            with parallel_backend("loky"):
                results = Parallel(
                    n_jobs=N_JOBS,
                    backend="loky",
                    prefer="processes",
                    verbose=PARALLEL_VERBOSE
                )(
                    delayed(tokenize_filter_and_save)(
                        chunk_texts, chunk_id, OUTPUT_DIR, TOKENIZER_MODEL, 20, LANG_MODE, ASCII_THRESHOLD
                    )
                    for chunk_texts, chunk_id in to_dispatch
                )

            # collect results immediately: results align with to_dispatch order
            for res in results:
                if res is None:
                    continue
                out_path, count, sample_tokens, sample_text = res
                saved_files.append(out_path)
                total_tokens += count
                # log sample
                short = sample_text if len(sample_text) <= 200 else sample_text[:200] + "…[truncated]"
                logger.info(f"Chunk saved: {out_path} — tokens={count} — sample tokens (first {len(sample_tokens)}): {sample_tokens}")
                logger.info(f"Chunk decoded sample -> {repr(short)}")

            # light GC
            gc.collect()

        # Fallback discovery in case we salvaged/created files but results list is empty
        if not saved_files:
            saved_files = [str(p) for p in Path(OUTPUT_DIR).glob("chunk_*.npy")]

        if not saved_files:
            logger.info("No chunk files produced.")
            return

        # numeric sort by chunk id extracted from filename
        saved_files.sort(key=lambda f: int(re.sub(r"\D", "", Path(f).stem)))
        logger.info(f"Produced {len(saved_files)} chunk files with total tokens={total_tokens}")

        merge_chunks(saved_files, MERGED_OUTPUT)
        logger.info(f"Final merged array at: {MERGED_OUTPUT}")

    except Exception:
        logger.exception("Pipeline failed")
        raise


if __name__ == "__main__":
    start = time.time()
    run_pipeline()
    logger.info(f"All done in {time.time()-start:.1f}s")


08:29:53 [INFO] Logging to file: logs\extract_1756016993.log
08:29:53 [INFO] Starting pipeline (Option B): INPUT=datasets/4GB, OUTPUT=outputs/extract_v11, N_JOBS=16
08:29:53 [INFO] Using string column: text
08:30:00 [INFO] Dispatching 64 chunks to workers (n_jobs=16)
[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done   9 tasks      | elapsed:  1.8min
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:  3.2min
[Parallel(n_jobs=16)]: Done  29 tasks      | elapsed:  3.6min
[Parallel(n_jobs=16)]: Done  40 out of  64 | elapsed:  5.2min remaining:  3.1min
[Parallel(n_jobs=16)]: Done  47 out of  64 | elapsed:  5.6min remaining:  2.0min
[Parallel(n_jobs=16)]: Done  54 out of  64 | elapsed:  6.8min remaining:  1.3min
[Parallel(n_jobs=16)]: Done  61 out of  64 | elapsed:  6.9min remaining:   20.4s
[Parallel(n_jobs=16)]: Done  64 out of  64 | elapsed:  7.0min finished
08:37:01 [INFO] Chunk saved: outputs\extract_v11\chunk_000000.npy — tokens

In [1]:
#!/usr/bin/env python3
"""
Parallel parquet -> filter (in-worker) -> tokenize -> chunk-save pipeline.

Concept / intent
----------------
This script streams many parquet files (FineWebEdu / 100BT style) without loading
them all into memory, groups rows into approximate byte-sized "chunks", and
dispatches those chunks to worker processes. Each worker:

  1. Filters rows for English (langid + ASCII-fallback),
  2. Tokenizes rows *per-row* using a fast BPE tokenizer (tiktoken),
  3. Optionally appends an End-Of-Sequence (EOS) token after each row,
  4. Writes out chunk token arrays atomically to disk (.npy),
  5. Returns a small sample (first N tokens and decoded text) for verification.

Why these design choices?
- Stream parquet with pyarrow: parquet files can be large; streaming keeps memory low.
- Chunk by approximate bytes: rows vary widely in length; bytes give better chunk balancing.
- Filter in-worker: this version filters inside workers so both language-detection and
  tokenization run in parallel (useful when langid is a CPU bottleneck).
- Per-row tokenization (not a single joined encode): it makes EOS insertion explicit and
  avoids ambiguity about token boundaries.
- Atomic save (temp -> os.replace): makes partial-run interruption safe and resumable.
- Salvage routine: repairs leftover temp files from prior buggy runs.
- Joblib loky backend: robust process-based parallelism for CPU-bound tasks.

Important: tune TARGET_CHUNK_BYTES, N_JOBS, PREFETCH_FACTOR to your machine.
"""

from __future__ import annotations

import os
import time
import uuid
import logging
import re
import gc
from pathlib import Path
from typing import List, Tuple, Optional, Iterator

import numpy as np
from joblib import Parallel, delayed, parallel_backend

import pyarrow.dataset as ds
import pyarrow as pa

# tiktoken / langid: fast BPE + language detection
import langid
import tiktoken

# ----------------- Configuration -----------------
# Paths
INPUT_DIR = "datasets/4GB"                   # directory with parquet files
PARQUET_GLOB = "*.parquet"                   # file glob for parquet files
OUTPUT_DIR = Path("outputs/extract_v11")     # where chunk .npy files go
MERGED_OUTPUT = OUTPUT_DIR / "tokens_merged.npy"

# Chunking / streaming
BATCH_ROWS = 4096                             # rows per pyarrow RecordBatch
TARGET_CHUNK_BYTES = 50_000_000               # approximate bytes per dispatched chunk

# Parallelism / joblib
TOKENIZER_MODEL = "gpt2"
N_JOBS = max(1, min(os.cpu_count() or 1, 128)) # processes (defaults to number of logical CPUs)
PREFETCH_FACTOR = 4                            # how many chunks to prefetch
PARALLEL_VERBOSE = 5                           # joblib verbose (0 / 5 / 10)

# Language detection
ASCII_THRESHOLD = 0.5
LANG_MODE = "accuracy"

# EOS handling (important for autoregressive pretraining)
# If True: append the tokenizer's EOS token ID after every input row.
ADD_EOS = True

# Tunables for atomic saves
ATOMIC_SAVE_MAX_RETRIES = 8
ATOMIC_SAVE_RETRY_DELAY = 0.2                 # seconds

# Ensure output dir
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------- Logging -----------------
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
run_id = int(time.time())
log_file = LOG_DIR / f"extract_{run_id}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

# Add file handler for persistent logs
file_handler = logging.FileHandler(str(log_file), mode="w", encoding="utf-8")
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", "%H:%M:%S"))
logger.addHandler(file_handler)
logger.info(f"Starting run; writing logs to {log_file}")

# ----------------- Helper utilities -----------------


def salvage_stale_temp_files(out_dir: Path) -> int:
    """
    Repair leftover temp artifacts from older buggy runs (np.save behavior).
    Handles both "chunk_*.npy.tmp.npy" and "chunk_*.npy.tmp" patterns.
    Returns number of renamed / repaired files.

    Why: historical bug where we passed a temp path string to np.save and numpy
    appended ".npy" automatically — leaving ".npy.tmp.npy" files behind.
    """
    repaired = 0
    # Pattern 1: chunk_XXXX.npy.tmp.npy -> chunk_XXXX.npy
    for p in out_dir.glob("chunk_*.npy.tmp.npy"):
        final = Path(str(p).replace(".npy.tmp.npy", ".npy"))
        if not final.exists():
            try:
                os.replace(str(p), str(final))
                repaired += 1
            except Exception as e:
                logger.debug(f"Could not salvage {p}: {e}")

    # Pattern 2: chunk_XXXX.npy.tmp -> chunk_XXXX.npy
    for p in out_dir.glob("chunk_*.npy.tmp"):
        final = p.with_suffix("")  # drops the final '.tmp' suffix
        if not final.exists():
            try:
                os.replace(str(p), str(final))
                repaired += 1
            except Exception as e:
                logger.debug(f"Could not salvage {p}: {e}")

    if repaired:
        logger.info(f"Salvaged {repaired} stale temp files in {out_dir}")
    return repaired


def list_parquet_files(input_dir: str, pattern: str = "*.parquet") -> List[str]:
    """Return sorted list of parquet file paths for input_dir/pattern."""
    p = Path(input_dir)
    if not p.exists():
        raise FileNotFoundError(f"Input dir not found: {p}")
    return sorted(str(x) for x in p.glob(pattern))


def choose_string_column(dataset: ds.dataset) -> str:
    """
    Pick the best string column name in the dataset schema.
    Prefers common names ('text','content','body') and falls back to
    the first string column found.
    """
    schema = dataset.schema
    str_columns = [f.name for f in schema if pa.types.is_string(f.type)]
    for candidate in ("text", "content", "body"):
        if candidate in str_columns:
            return candidate
    if not str_columns:
        raise RuntimeError("No string column in parquet schema(s)")
    return str_columns[0]


def is_english_text(text: str, mode: str = "accuracy", ascii_threshold: float = 0.5) -> bool:
    """
    Heuristic language detection used in this pipeline.

    Mode 'accuracy': prefer langid.classify (statistical detector). If langid
    does not say 'en' then fall back to an ASCII-fraction threshold. The
    fallback helps with short/dirty fragments where langid can be flaky.

    Returns True for detected English-like strings, False for others.
    """
    if not text:
        return False
    if mode == "accuracy":
        try:
            lang, _score = langid.classify(text)
        except Exception:
            lang = None
        if lang == "en":
            return True
        ascii_count = sum(1 for c in text if ord(c) < 128)
        total = len(text)
        if total == 0:
            return False
        return (ascii_count / total) >= ascii_threshold
    else:
        ascii_count = sum(1 for c in text if ord(c) < 128)
        return (ascii_count / max(1, len(text))) >= ascii_threshold


# ----------------- Streaming / chunking (main process) -----------------


def iter_raw_text_chunks_from_parquets(
    input_dir: str,
    pattern: str = "*.parquet",
    batch_rows: int = 4096,
    target_chunk_bytes: int = 50_000_000,
) -> Iterator[Tuple[List[str], int]]:
    """
    Stream parquet rows and yield chunks of raw rows (list[str], chunk_id).

    Important design:
      - We DO NOT filter in this generator for English. Filtering will be
        done inside worker processes so language detection runs in parallel.
      - We collect rows until the approximate UTF-8 byte-size of the buffer
        reaches target_chunk_bytes, which balances work per-worker better
        than fixed row counts when row lengths vary.

    Yields:
      - (list_of_rows, chunk_id), where chunk_id is a monotonically increasing integer.
    """
    file_list = list_parquet_files(input_dir, pattern)
    if not file_list:
        logger.warning("No parquet files found for pattern %s in %s", pattern, input_dir)
        return

    dataset = ds.dataset(file_list, format="parquet")
    chosen_col = choose_string_column(dataset)
    logger.info("Using string column '%s' for text extraction", chosen_col)

    scanner = dataset.scanner(batch_size=batch_rows)
    chunk_id = 0
    buffer_texts: List[str] = []
    buffer_bytes = 0

    for record_batch in scanner.to_batches():
        # Get the named column from the RecordBatch (fast)
        arr = record_batch[chosen_col]
        for raw in arr.to_pylist():
            if raw is None:
                continue
            text = str(raw).strip()
            if not text:
                continue
            b = len(text.encode("utf-8"))

            # If adding this row would exceed the target and we already have
            # some buffered rows, yield the buffer as a dispatchable chunk.
            if buffer_texts and (buffer_bytes + b) >= target_chunk_bytes:
                yield buffer_texts, chunk_id
                chunk_id += 1
                buffer_texts = []
                buffer_bytes = 0

            # If a single row is larger than the target chunk, emit it alone
            # (we don't want to infinite-loop attempting to pack it).
            if b >= target_chunk_bytes:
                yield [text], chunk_id
                chunk_id += 1
                continue

            # Otherwise append to buffer
            buffer_texts.append(text)
            buffer_bytes += b

    # Flush any remaining buffered rows
    if buffer_texts:
        yield buffer_texts, chunk_id


# ----------------- Worker: filter (in-worker) + tokenize + save -----------------


def _safe_atomic_save_npy(arr: np.ndarray, out_path: Path, max_retries: int = ATOMIC_SAVE_MAX_RETRIES,
                          retry_delay: float = ATOMIC_SAVE_RETRY_DELAY) -> None:
    """
    Safely and atomically write a numpy array to `out_path`.

    Implementation details:
      - Create a uniquely-named temporary file in the same directory (so os.replace
        is atomic on the same filesystem).
      - Open a file handle and pass it to np.save(...) *to avoid numpy appending '.npy'*
        to our chosen tmp filename.
      - Call os.replace(tmp, final) to atomically publish the final file.
      - Retry a few times to tolerate transient Windows AV locks or network FS issues.
    """
    out_dir = out_path.parent
    out_dir.mkdir(parents=True, exist_ok=True)

    tmp_name = f"{out_path.stem}.{os.getpid()}.{uuid.uuid4().hex}.npy.tmp"
    tmp_path = out_dir / tmp_name

    # Write using file handle to prevent numpy from appending extra suffixes.
    with open(tmp_path, "wb") as fh:
        # np.save accepts file-like objects; this preserves the exact tmp filename.
        np.save(fh, arr)

    if not tmp_path.exists():
        raise FileNotFoundError(f"Temp write failed: {tmp_path}")

    last_exc = None
    for attempt in range(1, max_retries + 1):
        try:
            os.replace(str(tmp_path), str(out_path))
            return
        except (FileNotFoundError, PermissionError) as e:
            last_exc = e
            time.sleep(retry_delay)
        except Exception:
            # unexpected — re-raise
            raise
    raise last_exc or RuntimeError("Atomic replace failed for unknown reason")


def _get_eos_token_id(enc: "tiktoken.Encoding") -> Optional[int]:
    """
    Return the tokenizer's canonical end-of-sequence token id, if available.

    Strategy:
      - Try to encode the special string that GPT-2 uses for end-of-text: ""
        using the tokenizer's encode with allowed special tokens (if available).
      - Fall back to known GPT-2 EOS id (50256) for the 'gpt2' model.
      - If we still can't determine one, return None and downstream code will
        skip EOS insertion (and log a warning).
    """
    # Try the official token string: "" — many OpenAI GPT2 encoders will map this.
    try:
        # prefer the general encode (may accept allowed_special sets)
        ids = enc.encode("", allowed_special={"<|endoftext|>"})
        if ids:
            return int(ids[0])
    except Exception:
        # sometimes encode() doesn't accept the allowed_special kwarg; try plain encode
        try:
            ids = enc.encode("")
            if ids:
                return int(ids[0])
        except Exception:
            pass

    # Known fallback: GPT-2 uses token id 50256 for ""
    try:
        if TOKENIZER_MODEL.lower().startswith("gpt2"):
            return 50256
    except Exception:
        pass

    # Could not determine EOS id
    return None


def tokenize_filter_and_save(
    chunk_texts: List[str],
    chunk_id: int,
    out_dir: str,
    tokenizer_model: str,
    sample_n: int = 20,
    lang_mode: str = LANG_MODE,
    ascii_threshold: float = ASCII_THRESHOLD,
    add_eos: bool = ADD_EOS
) -> Optional[Tuple[str, int, List[int], str]]:
    """
    Worker function executed inside a separate process.

    Steps:
      1. Instantiate a tiktoken encoder in the worker process (avoids pickling).
      2. Filter chunk_texts for English with is_english_text (running in parallel).
      3. Tokenize each retained row separately, optionally append EOS token id
         after each row (makes document boundaries explicit).
      4. Save tokens to disk atomically.
      5. Return (final_path, token_count, sample_tokens, decoded_sample_text)
         or None if this chunk produced nothing.

    Note on design:
      - Per-row tokenization gives precise control over boundaries (EOS insertion).
      - If add_eos is False we produce a plain concatenation of tokens.
    """
    # Create tokenizer instance inside worker — faster and avoids pickling
    try:
        try:
            enc = tiktoken.get_encoding(tokenizer_model)
        except Exception:
            enc = tiktoken.encoding_for_model(tokenizer_model)
    except Exception as e:
        logger.exception("Worker failed to create tokenizer: %s", e)
        return None

    # Determine EOS id once per worker invocation (cheap)
    eos_id = _get_eos_token_id(enc) if add_eos else None
    if add_eos and eos_id is None:
        logger.warning("ADD_EOS requested but EOS token id couldn't be determined; EOS will be skipped for this run.")

    # Filter rows locally (parallelizes the CPU-heavy language detection)
    kept_rows = []
    for txt in chunk_texts:
        if not txt:
            continue
        if is_english_text(txt, mode=lang_mode, ascii_threshold=ascii_threshold):
            kept_rows.append(txt)
    if not kept_rows:
        # nothing to save for this chunk
        return None

    # Tokenize per row so EOS insertion is unambiguous
    token_ids: List[int] = []
    for row in kept_rows:
        try:
            toks = enc.encode_ordinary(row)
        except Exception:
            toks = enc.encode(row)
        if toks:
            token_ids.extend(toks)
            # Append EOS token id if configured and available
            if eos_id is not None:
                token_ids.append(eos_id)

    if not token_ids:
        return None

    arr = np.array(token_ids, dtype=np.int32)
    out_path = Path(out_dir) / f"chunk_{chunk_id:06d}.npy"

    # Resumability: skip if already exists
    if out_path.exists():
        logger.info("Chunk %s exists, skipping write", out_path.name)
    else:
        # atomic save with retries
        try:
            _safe_atomic_save_npy(arr, out_path)
        except Exception as e:
            logger.exception("Failed to save chunk %d -> %s : %s", chunk_id, out_path, e)
            # best-effort cleanup of any temp variants
            try:
                stem = out_path.stem
                for pat in (f"{stem}.*.npy.tmp", f"{stem}.*.npy.tmp.npy", f"{stem}.*.tmp"):
                    for f in out_path.parent.glob(pat):
                        try:
                            f.unlink(missing_ok=True)
                        except Exception:
                            pass
            except Exception:
                pass
            return None

    # Prepare a human-check sample (first sample_n token ids and decoded text)
    sample_tokens = arr[:sample_n].tolist()
    try:
        sample_text = enc.decode(sample_tokens)
    except Exception:
        sample_text = " ".join(str(t) for t in sample_tokens)

    return str(out_path), int(arr.size), sample_tokens, sample_text


# ----------------- Merge chunk files (main process) -----------------


def merge_chunks(chunk_files: List[str], out_path: Path) -> None:
    """
    Merge numerically sorted chunk .npy files into one large memmapped int32 .npy.

    Behavior:
      - Uses np.memmap to avoid allocating the entire concatenated array in RAM.
      - Expects chunk_files to be sorted by chunk id.
    """
    if not chunk_files:
        raise ValueError("No chunk files provided to merge")

    total = 0
    lengths = []
    for f in chunk_files:
        arr = np.load(f, mmap_mode="r")
        lengths.append(arr.shape[0])
        total += arr.shape[0]

    dtype = np.int32
    merged = np.memmap(str(out_path), dtype=dtype, mode="w+", shape=(total,))
    cursor = 0
    for f in chunk_files:
        arr = np.load(f)
        n = arr.shape[0]
        merged[cursor: cursor + n] = arr
        cursor += n
    merged.flush()
    del merged
    logger.info("Merged %d chunks (%d tokens) -> %s", len(chunk_files), total, out_path)


# ----------------- Orchestration (main) -----------------


def run_pipeline():
    """
    Main orchestration:
      - Salvage stale temp files
      - Stream / chunk parquet rows
      - Dispatch chunks in batches to joblib workers
      - Collect samples and saved file list
      - Merge chunk files into final memmapped array
    """
    logger.info("Pipeline start: INPUT=%s OUTPUT=%s N_JOBS=%d", INPUT_DIR, OUTPUT_DIR, N_JOBS)

    # Repair leftovers (if any) to avoid accidental re-processing
    salvage_stale_temp_files(OUTPUT_DIR)

    chunk_gen = iter_raw_text_chunks_from_parquets(
        INPUT_DIR, PARQUET_GLOB, batch_rows=BATCH_ROWS, target_chunk_bytes=TARGET_CHUNK_BYTES
    )

    saved_files: List[str] = []
    total_tokens = 0
    max_pending = max(1, N_JOBS * PREFETCH_FACTOR)

    try:
        while True:
            # Prefetch up to max_pending chunks for dispatch
            to_dispatch: List[Tuple[List[str], int]] = []
            for _ in range(max_pending):
                try:
                    chunk_texts, chunk_id = next(chunk_gen)
                    to_dispatch.append((chunk_texts, chunk_id))
                except StopIteration:
                    break

            if not to_dispatch:
                break

            logger.info("Dispatching %d chunks to workers (n_jobs=%d)", len(to_dispatch), N_JOBS)
            with parallel_backend("loky"):
                results = Parallel(
                    n_jobs=N_JOBS,
                    backend="loky",
                    prefer="processes",
                    verbose=PARALLEL_VERBOSE
                )(
                    delayed(tokenize_filter_and_save)(
                        chunk_texts, chunk_id, str(OUTPUT_DIR), TOKENIZER_MODEL, 20, LANG_MODE, ASCII_THRESHOLD, ADD_EOS
                    )
                    for chunk_texts, chunk_id in to_dispatch
                )

            # collect results as they return
            for res in results:
                if res is None:
                    continue
                out_path, count, sample_tokens, sample_text = res
                saved_files.append(out_path)
                total_tokens += count
                short = sample_text if len(sample_text) <= 200 else sample_text[:200] + "…[truncated]"
                logger.info("Chunk saved: %s — tokens=%d — sample tokens=%s", out_path, count, sample_tokens)
                logger.info("Chunk decoded sample -> %s", repr(short))

            # light GC after each dispatch
            gc.collect()

        # If no results were returned (e.g. we salvaged existing files but results list empty),
        # fall back to scanning disk for chunk_*.npy
        if not saved_files:
            saved_files = [str(p) for p in OUTPUT_DIR.glob("chunk_*.npy")]

        if not saved_files:
            logger.info("No chunk files found/produced.")
            return

        # numeric sort and merge
        saved_files.sort(key=lambda f: int(re.sub(r"\D", "", Path(f).stem)))
        logger.info("Produced %d chunk files (total tokens approx %d). Starting merge...", len(saved_files), total_tokens)
        merge_chunks(saved_files, MERGED_OUTPUT)
        logger.info("Final merged array at: %s", MERGED_OUTPUT)

    except Exception:
        logger.exception("Pipeline failed")
        raise


if __name__ == "__main__":
    t0 = time.time()
    run_pipeline()
    logger.info("All done in %.1fs", time.time() - t0)


19:26:55 [INFO] Starting run; writing logs to logs\extract_1756056415.log
19:26:55 [INFO] Pipeline start: INPUT=datasets/4GB OUTPUT=outputs\extract_v11 N_JOBS=16
19:26:55 [INFO] Using string column 'text' for text extraction
19:27:02 [INFO] Dispatching 64 chunks to workers (n_jobs=16)
[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  46 out of  64 | elapsed:  5.5min remaining:  2.2min
[Parallel(n_jobs=16)]: Done  59 out of  64 | elapsed:  7.1min remaining:   35.9s
[Parallel(n_jobs=16)]: Done  64 out of  64 | elapsed:  7.2min finished
19:34:12 [INFO] Chunk saved: outputs\extract_v11\chunk_000000.npy — tokens=10850108 — sample tokens=[464, 13362, 12091, 198, 1890, 477, 262, 1842, 11, 19661, 290, 10731, 287, 12091, 2517, 268, 447, 247, 82, 3835]
19:34:12 [INFO] Chunk decoded sample -> 'The Independent Jane\nFor all the love, romance and scandal in Jane Austen’s books'
19:34:12 [INFO] Chunk saved: outputs\extract_v11\chunk_000001.npy 